# 04：一个问题，三种模拟表示

**适合谁：**已了解量子电路期望值和训练循环、希望理解 FlagQuantum 执行选项的用户。

**你的任务：**用同一个电路构造函数、能量函数和训练循环，分别运行态向量、MPS 和张量网络模拟。
比较训练过程前，先检查三种表示在相同参数下是否给出相同的能量与梯度。

这是正确性和 API 使用实验。我们故意使用很小的双量子比特问题，它不能证明有实际意义的性能排名。

[English edition](../notebooks/04_one_problem_three_simulators.ipynb)

**学习目标：** 切换模拟表示，同时保留同一个电路、目标函数与梯度检查。

**先修要求：** 已完成 02，了解电路期望值与基本矩阵记法。新人可以先用默认模式，之后再回来学习；不需要加速卡或云端账号。

学习路线见 [workshop 导航](../README.md)。


## 1. 只定义一次问题

最小化 H = -Z₀Z₁ - 0.5X₀ - 0.5X₁。对于这个小系统，稠密的 4×4 矩阵提供独立的精确能量参考。
下方电路和可观测量的定义都不指定模拟器。


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import torch
import flagquantum as fq

observable = -1.0 * (fq.Z(0) @ fq.Z(1)) - 0.5 * fq.X(0) - 0.5 * fq.X(1)
Z = torch.tensor([[1.0, 0.0], [0.0, -1.0]], dtype=torch.float64)
X = torch.tensor([[0.0, 1.0], [1.0, 0.0]], dtype=torch.float64)
I = torch.eye(2, dtype=torch.float64)
H = -torch.kron(Z, Z) - 0.5 * torch.kron(X, I) - 0.5 * torch.kron(I, X)
exact_energy = torch.linalg.eigvalsh(H).min().item()


def circuit(theta):
    q = fq.Circuit(2)
    q.ry(0, theta[0])
    q.ry(1, theta[1])
    q.cx(0, 1)
    q.ry(0, theta[2])
    q.ry(1, theta[3])
    return q


print("Exact ground energy:", exact_energy)


## 2. 将模拟表示作为执行选项

这个函数只有 mode 负责选择表示。关闭后端回退后，不支持的选择会明确报错，而不是悄悄换成另一个实现。
我们直接请求能量，因此优化循环无需转换为稠密态向量。

电路图中，每条横线对应一个量子比特，量子门从左到右执行。图中的上下顺序不等于云平台结果位串的排列顺序。 这里只画初始参数下的电路，不在训练循环中重复绘图。

**概念：** 态向量保存全部 2ⁿ 个复振幅。MPS 用一串张量表示量子态，在状态具有可压缩结构时可节省内存。张量网络通过收缩张量计算所需结果，成本取决于电路结构与收缩顺序。

**预期输出：** 这个小规模精确例子的三种模式应给出一致的能量和梯度。更大实验中的 MPS 键维数限制或截断可能引入近似，需要单独检查。

**常见误解：** 新用户不必一开始就选型。普通小电路继续使用默认模式；态向量内存不足且纠缠可控时尝试 MPS，电路结构与所需观测量适合时尝试张量网络。应实测自身任务的内存、精度和梯度，另外两种模式并不保证更快。


In [ ]:
def evaluate(theta, mode):
    return fq.run(
        circuit(theta),
        options=fq.ExecutionOptions(
            mode=mode, device="cpu", allow_backend_fallback=False
        ),
        outputs=fq.expectation(observable),
    )


modes = ["statevector", "mps", "tensor_network"]
torch.manual_seed(7)
initial_theta = 0.2 * torch.randn(4)
initial_circuit = circuit(initial_theta)
fig, ax = initial_circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
probes = {}
for mode in modes:
    theta = initial_theta.clone().requires_grad_(True)
    result = evaluate(theta, mode)
    energy = result.expectation().sum()
    (gradient,) = torch.autograd.grad(energy, theta)
    probes[mode] = {
        "energy": energy.detach(),
        "gradient": gradient.detach(),
        "runtime": dict(result.runtime),
    }
    print(mode, "energy:", energy.item())


## 3. 检查数值和梯度是否一致

只在一个点得到相同能量，还不足以保证训练一致：不同的梯度会让优化器走向不同方向。
下方容差允许这个 CPU 示例中出现小幅浮点差异。


In [ ]:
reference = probes["statevector"]
for mode in modes[1:]:
    torch.testing.assert_close(
        probes[mode]["energy"], reference["energy"], atol=1e-5, rtol=1e-5
    )
    torch.testing.assert_close(
        probes[mode]["gradient"], reference["gradient"], atol=1e-5, rtol=1e-5
    )
    print(
        mode,
        "maximum gradient difference:",
        (probes[mode]["gradient"] - reference["gradient"]).abs().max().item(),
    )


## 4. 复用同一个训练循环

每次运行都从相同参数开始，并创建新的优化器。不同表示之间不要复用优化器的动量状态。
每次训练都使用相同的 160 步预算。


In [ ]:
def train(mode):
    theta = torch.nn.Parameter(initial_theta.clone())
    optimizer = torch.optim.Adam([theta], lr=0.08)
    history = []
    for step in range(160):
        optimizer.zero_grad()
        energy = evaluate(theta, mode).expectation().sum()
        energy.backward()
        optimizer.step()
        history.append(energy.detach().item())
    final_energy = evaluate(theta, mode).expectation().item()
    return {
        "history": history,
        "final_energy": final_energy,
        "parameters": theta.detach().tolist(),
    }


runs = {mode: train(mode) for mode in modes}
for mode, run in runs.items():
    gap = run["final_energy"] - exact_energy
    print(mode, "final energy:", run["final_energy"], "gap:", gap)
    assert abs(gap) < 0.02


## 5. 比较学习曲线

三条曲线应接近相同的能量。由于不同表示可能以不同顺序进行浮点运算，小差异是正常的。
这里证明了同一程序和目标可以复用，并不意味着各表示的速度或内存开销相同。

**读懂输出：** 三种模式的能量应与态向量参考值一致，`maximum gradient difference` 在给定容差内应接近零。各模式最终能量应与 `Exact ground energy` 相差小于 0.02。答案一致不意味着内存或速度一样。

**安全重复运行：** 每次调用 `train(mode)` 都会创建新的参数和优化器。修改电路或目标后，应先重新运行独立参考值与梯度检查。


In [ ]:
for mode, run in runs.items():
    plt.plot(run["history"], label=mode)
plt.axhline(exact_energy, color="black", linestyle="--", label="Exact ground energy")
plt.xlabel("Optimizer step")
plt.ylabel("Energy")
plt.legend()
plt.show()
assert (
    max(r["final_energy"] for r in runs.values())
    - min(r["final_energy"] for r in runs.values())
    < 2e-4
)


## 6. 保存比较结果


In [ ]:
from pathlib import Path

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
WORKSHOP = next(
    (root / "flagos2026" for root in search_roots if (root / "flagos2026/notebooks").is_dir()),
    None,
)
if WORKSHOP is None:
    raise RuntimeError("Cannot locate flagos2026/notebooks above the current working directory. Open this notebook from the deployed workshop tree.")
repository = WORKSHOP.parent.parent
ROOT = repository if (repository / "pyproject.toml").is_file() else WORKSHOP
OUTPUTS = WORKSHOP / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)


In [ ]:
import json

report = {
    "experiment": "one_objective_three_representations",
    "source": "local_simulation",
    "flagquantum_version": fq.__version__,
    "torch_version": torch.__version__,
    "seed": 7,
    "steps": 160,
    "learning_rate": 0.08,
    "qubits": 2,
    "initial_parameters": initial_theta.tolist(),
    "exact_energy": exact_energy,
    "final_energies": {mode: run["final_energy"] for mode, run in runs.items()},
    "fallback_allowed": False,
    "scope": "CPU correctness and gradient parity; no performance or scalability claim",
}
(OUTPUTS / "three_simulators_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))


## 向别人解释这个实验

- 哪些代码描述科学问题，哪些代码选择执行表示？
- 梯度比较说明了什么，是只比较最终能量无法说明的？
- 为什么增加量子比特数量后需要重新实验，不能直接根据这里的耗时推断？

**接下来尝试：**同时修改可观测量和参考矩阵中的外场强度，重新训练前先重复一致性检查。
然后参考[扩展示例 11](../reference_notebooks/simulation/11_mps.ipynb)，通过显式键维上限研究近似误差和存储。
用[实验记录](../FIELD_NOTES.md)写下观察。


## 带回家使用

保留同一个电路和可观测量，通过 `fq.ExecutionOptions` 切换执行表示。
这个独立的前向计算示例不使用 workshop 辅助代码。

```python
import flagquantum as fq

circuit = fq.Circuit(2).h(0).cx(0, 1)
observable = fq.Z(0) @ fq.Z(1)
for mode in ("statevector", "mps", "tensor_network"):
    result = fq.run(
        circuit,
        options=fq.ExecutionOptions(
            mode=mode, device="cpu", allow_backend_fallback=False,
        ),
        outputs=fq.expectation(observable),
    )
    print(mode, result.expectation().item())  # approximately 1
```

小规模 CPU 问题的结果一致，不能证明性能相同或具备分布式扩展能力。
修改线路结构或目标时，应保留梯度一致性与独立参考检查。
